# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/faith-amanze/content-refresh-prioritization/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

**My lane:** Content Refresh Prioritization, framed (w02) as a scoring/ranking task — the
deliverable is an ordered review queue, not a plain label. Success metric: **Precision@50**.

**Target:** I reuse the pipeline's `is_declining_label` (1 when `trend_direction == "down"`),
which is the same decline/staleness signal my w02 framing pointed at. It's an *observed*
label built from real trend data, not a rule I'm inventing — 54.2% of rows are positive.

**Method:** Per the `training-honest-models` skill, a "yes/no with an observed label" question
that will be *evaluated as a ranking* (probability, cut at K) starts with **Logistic
Regression, then Random Forest** — readable first, stronger second. I also add a shallow
**Decision Tree** (depth 4) as a middle point: still printable and explainable, and useful
for sanity-checking whether the story a linear model tells is stable under a nonlinear model.

I did **not** reach for Gradient Boosting: with 32 clients and one train/test split, a GBM's
extra flexibility is more likely to overfit noise than to earn its complexity here — the skill
note "add complexity only when the comparison earns it" is doing the work in this decision.

**Leakage note:** `trend_direction` / `trend_pct` are the label's source and are excluded from
features (per the flyrank-data skill's label trap). I also exclude `impressions_last_30d`,
`impressions_prev_30d`, and the matching clicks/sessions last-30/prev-30 columns — the label
is *literally* last-30 vs prev-30, so those columns would let a model reconstruct the label
almost exactly rather than learn a generalizable signal. `content_id`/`client_id` are used for
grouping only, never as features.

In [1]:
# ── Setup: load data, define target, define leakage-safe feature list ──
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score, average_precision_score

RANDOM_STATE = 42
pd.set_option("display.max_colwidth", 60)

# Path resolution: works in Codespaces (/workspaces/assignment1/...) and other clones alike.
_candidates = [
    "/workspaces/assignment1/data/raw/content_refresh_anonymized.csv",
    "../../data/raw/content_refresh_anonymized.csv",
    "data/raw/content_refresh_anonymized.csv",
]
import os
_data_path = next(p for p in _candidates if os.path.exists(p))
df = pd.read_csv(_data_path)

# Target: reuses the same decline/staleness signal I framed in w02 (ML-03) —
# "is_declining_label" is the pipeline's standard definition:
# 1 when trend_direction == "down", built directly from the flyrank-data skill's label.
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
print(f"Rows: {len(df):,} | positive rate (declining): {df['is_declining_label'].mean():.3f}")

# Leakage-safe numeric features: everything that describes CURRENT state, none of it
# derived from the label. Excluded on purpose (see leakage note below):
#   trend_direction, trend_pct        -> these ARE the label / define the label
#   impressions_last_30d, prev_30d,
#   clicks_last_30d/prev_30d,
#   sessions_last_30d/prev_30d        -> the label is a function of last_30 vs prev_30,
#                                         so these would let the model reconstruct the
#                                         label almost exactly. Same leakage family.
#   content_id, client_id             -> pseudonyms, grouping only, never features
#   provider_used, model_used         -> data dictionary marks these "not a model feature"
#   *_tier / *_tier_order columns     -> redundant restatements of columns already included
numeric_features = [
    "avg_position", "impressions_90d", "clicks_90d", "ctr",
    "pageviews_90d", "sessions_90d", "users_90d", "engaged_sessions_90d",
    "ai_sessions_90d", "scroll_events_90d", "days_with_impressions",
    "days_with_sessions", "engagement_rate", "scroll_rate", "ai_traffic_pct",
    "word_count", "char_count", "content_age_days", "days_since_last_update",
    "search_volume", "competition", "cpc",
]
categorical_features = ["content_type", "main_intent", "competition_level"]

# Missingness follows content_type (per data-dictionary) -> add has_-flags instead of
# a blind fillna, so "missing" stays visible to the model as its own signal.
df["has_keyword_data"] = df["search_volume"].notna().astype(int)
df["has_word_count"] = df["word_count"].notna().astype(int)
df["has_position_data"] = (df["avg_position"] > 0).astype(int)
numeric_features += ["has_keyword_data", "has_word_count", "has_position_data"]

feature_cols = numeric_features + categorical_features
X = df[feature_cols].copy()
y = df["is_declining_label"].copy()
groups = df["client_id"]

print(f"Feature matrix: {X.shape}, using {len(numeric_features)} numeric + {len(categorical_features)} categorical columns")


Rows: 30,000 | positive rate (declining): 0.542
Feature matrix: (30000, 28), using 25 numeric + 3 categorical columns


## 2. Split design

**Grouped by client, same design as my Week-4 baseline.** `client_id` has 32 distinct pseudonym
values (grouping/joins only, per the flyrank-data skill — never a feature). If I split by row,
the same client's pages could land in both train and test, and the model could partly learn
"this is client X's typical CTR/position profile" instead of a signal that generalizes to a
client it has never seen. `GroupShuffleSplit` keeps every client entirely on one side, a 75/25
split by rows, seeded for reproducibility. I verify zero client overlap below.

In [2]:
# ── Grouped, client-holdout split (same design as my w04 baseline notebook) ──
# content_id/client_id are pseudonyms per the flyrank-data skill: fine for grouping,
# never as features. Splitting randomly by ROW would let the same client appear in
# both train and test, so the model could learn client-specific quirks (a client's
# typical CTR, its content mix) rather than a generalizable "is this declining" signal.
# GroupShuffleSplit keeps every client entirely on one side of the split.
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X, y, groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

overlap = set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])
print(f"Train: {len(X_train):,} rows, {groups.iloc[train_idx].nunique()} clients")
print(f"Test:  {len(X_test):,} rows, {groups.iloc[test_idx].nunique()} clients")
print(f"Client overlap between train/test: {len(overlap)} (must be 0)")
print(f"Train positive rate: {y_train.mean():.3f} | Test positive rate: {y_test.mean():.3f}")


Train: 22,885 rows, 24 clients
Test:  7,115 rows, 8 clients
Client overlap between train/test: 0 (must be 0)
Train positive rate: 0.550 | Test positive rate: 0.517


## 3. Train + compare vs my baseline

Same test rows, same metric (Precision@50, from my w02 framing), same split as above. The
baseline is my Week-4 rule (`weak_position * has_visibility * impressions_90d`, ranking pages
with a weak position but real search visibility) — recomputed here on the identical held-out
test rows so the comparison is apples-to-apples, not two different samples.

In [3]:
# ── Preprocessing pipelines ──
preprocess_scaled = ColumnTransformer([
    ("num", Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]), numeric_features),
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="constant", fill_value="unknown")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), categorical_features),
])
preprocess_tree = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), numeric_features),
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="constant", fill_value="unknown")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), categorical_features),
])

log_reg = Pipeline([("prep", preprocess_scaled),
    ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE))])
dtree = Pipeline([("prep", preprocess_tree),
    ("clf", DecisionTreeClassifier(max_depth=4, min_samples_leaf=50,
                                    class_weight="balanced", random_state=RANDOM_STATE))])
rf = Pipeline([("prep", preprocess_tree),
    ("clf", RandomForestClassifier(n_estimators=300, max_depth=8, min_samples_leaf=20,
                                    class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1))])

def precision_at_k(y_true, scores, k=50):
    """Precision@K: of the top-K ranked rows, what fraction are actually positive?
    Matches the Precision@50 success metric from my w02 framing notebook."""
    order = np.argsort(-scores)[:k]
    y_arr = y_true.values if hasattr(y_true, "values") else y_true
    return y_arr[order].mean()

results = {}

log_reg.fit(X_train, y_train)
results["logistic_regression"] = log_reg.predict_proba(X_test)[:, 1]

dtree.fit(X_train, y_train)
results["decision_tree"] = dtree.predict_proba(X_test)[:, 1]

rf.fit(X_train, y_train)
results["random_forest"] = rf.predict_proba(X_test)[:, 1]

# ── Baseline: my w04 rule, recomputed on the SAME test rows / same split ──
df_test = df.iloc[test_idx].copy()
weak_position = (df_test["avg_position"] >= 11).astype(int)
has_visibility = (df_test["impressions_90d"] >= 500).astype(int)
results["baseline_rule (w04)"] = (weak_position * has_visibility * df_test["impressions_90d"]).values

# ── The comparison table ──
rows = []
for name, scores in results.items():
    p50 = precision_at_k(y_test, scores, k=50)
    auc = roc_auc_score(y_test, scores)
    ap = average_precision_score(y_test, scores)
    rows.append({"model": name, "precision@50": round(p50, 3),
                 "roc_auc": round(auc, 3), "avg_precision": round(ap, 3)})
comparison = pd.DataFrame(rows).sort_values("precision@50", ascending=False).reset_index(drop=True)
comparison.loc[len(comparison)] = ["base_rate (test set)", round(y_test.mean(), 3), None, None]
comparison


,model,precision@50,roc_auc,avg_precision
0,logistic_regression,0.740,0.6,0.605
1,decision_tree,0.640,0.597,0.582
2,random_forest,0.500,0.608,0.591
3,baseline_rule (w04),0.480,0.486,0.507
4,base_rate (test set),0.517,None,None


## 4. Errors and interpretation

**Permutation importance** on the winning model (logistic regression), scored on average
precision — a ranking-quality metric, matching the actual task. Then a look at the top-50 the
model actually returns: how many are wrong, and what do the wrong ones have in common?

In [4]:
# ── Permutation importance for the winning model (logistic_regression) ──
# Scored on average_precision (a ranking-quality metric) rather than accuracy,
# since the real task is ranking a review queue, not a plain yes/no call.
perm = permutation_importance(
    log_reg, X_test, y_test, scoring="average_precision",
    n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1,
)
importance_table = pd.DataFrame({
    "feature": X_test.columns,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std,
}).sort_values("importance_mean", ascending=False).reset_index(drop=True)
print("Top 10 features by permutation importance:")
print(importance_table.head(10).to_string(index=False))

# ── Where is the model wrong? Look at the top-50 it actually returns ──
lr_scores = results["logistic_regression"]
order = np.argsort(-lr_scores)[:50]
top50 = df_test.iloc[order].copy()
top50["predicted_prob"] = lr_scores[order]
top50["actual_label"] = y_test.values[order]

n_wrong = int((top50["actual_label"] == 0).sum())
print(f"\n{n_wrong} of the top 50 ranked rows were NOT actually declining (false positives).")
print("\nA sample of the wrong ones:")
wrong_cols = ["content_id", "avg_position", "impressions_90d", "ctr", "trend_direction", "predicted_prob"]
top50.loc[top50["actual_label"] == 0, wrong_cols].head(6)


Top 10 features by permutation importance:
              feature  importance_mean  importance_std
days_with_impressions         0.049308        0.002946
   days_with_sessions         0.027582        0.002422
         avg_position         0.022476        0.002584
            users_90d         0.020361        0.001811
     content_age_days         0.018541        0.002172
          scroll_rate         0.008256        0.004023
         sessions_90d         0.007656        0.003170
    competition_level         0.004457        0.003074
           char_count         0.004188        0.001167
    has_position_data         0.003202        0.001147

13 of the top 50 ranked rows were NOT actually declining (false positives).

A sample of the wrong ones:


,content_id,avg_position,impressions_90d,ctr,trend_direction,predicted_prob
10175,content_374e795aab68,31.0,235,0.85,stable,0.888801
27993,content_26d48a980581,4.6,1266,0.00,up,0.879968
26614,content_7be5f150dc65,5.9,290,0.00,up,0.879134
28293,content_619acf4bbcc3,7.2,181,0.00,up,0.877406
8016,content_c94a53e3bfb8,8.1,2164,0.23,up,0.875871
11828,content_901bf2676f2a,7.4,989,0.00,up,0.875756


### What the results say

**Model vs baseline:** all three learned models beat the Week-4 rule at Precision@50 on the
held-out clients — logistic regression: 0.740 vs baseline 0.480 (base rate 0.517). That's a real
gain, but note it's *not* monotonic with model complexity: logistic regression beats the deeper
decision tree, which beats random forest, on Precision@50 — even though random forest has the
best ROC AUC. That mismatch is itself the finding from the `training-honest-models` skill note:
"if the model wins at one metric but loses at another — report both." A ranking metric evaluated
on only the top 50 rows is high-variance and reflects a different part of the score distribution
than an overall AUC; more flexible models here overfit to structure that helps *ranking in
general* but doesn't win the very top of the queue on this held-out client set.

**Top features (permutation importance):** `days_with_impressions`, `days_with_sessions`, and
`avg_position` dominate — the model leans on how *consistently* a page shows up and ranks, not
just raw traffic totals. This makes editorial sense: a page trending down usually shows fewer
active days before it shows fewer total impressions.

**Where it's wrong:** roughly a quarter of the top-50 queue were false positives. Looking at
them, several share a pattern: `avg_position` in the 4-9 range (already ranking *well*) and
`ctr = 0.00`, but `trend_direction` is actually "up" or "stable." My guess: pages with very low
absolute traffic (a handful of impressions) can swing to ctr=0.00 by chance in a 90-day window
without truly declining — the model is reading "zero clicks" as a decline signal when it's
really a low-volume/noise page. A useful next step would be adding an impressions floor before
trusting a `ctr = 0` reading, similar to how the baseline already used an `impressions_90d >= 500`
gate.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
